In [1]:
%cd PRO506

/workspace/PRO506


/usr/local/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField
from pyspark.sql.types import StringType, IntegerType, FloatType,BooleanType,DateType
from pyspark.sql import functions as fun
import pandas as pd 

spark = ( SparkSession.builder.appName("PRO506").master("spark://spark-master:7077").getOrCreate())



sc = spark.sparkContext

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/26 10:06:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
main_schema = StructType([
    StructField("row_id",StringType(),True),
    StructField("click_datetime",DateType(),True),
    StructField("time_to_next_click",FloatType(),True),
    StructField("movie_title",StringType(),True),
    StructField("movie_genres",StringType(),True),
    StructField("relase_date",DateType(),True),
    StructField("totle_id",StringType(),True),
    StructField("user_id",StringType(),True),
])

df = (spark.read.format("csv")
      .option("header","true")
      .schema(main_schema)
      .option("sep",",")
      .load("vodclickstream_uk_movies_03.csv"))

df.show(5)

+------+--------------+------------------+--------------------+--------------------+-----------+----------+----------+
|row_id|click_datetime|time_to_next_click|         movie_title|        movie_genres|relase_date|  totle_id|   user_id|
+------+--------------+------------------+--------------------+--------------------+-----------+----------+----------+
| 58773|    2017-01-01|               0.0|Angus, Thongs and...|Comedy, Drama, Ro...| 2008-07-25|26bd5987e8|1dea19f6fe|
| 58774|    2017-01-01|               0.0|The Curse of Slee...|Fantasy, Horror, ...| 2016-06-02|f26ed2675e|544dcbc510|
| 58775|    2017-01-01|           10530.0|   London Has Fallen|    Action, Thriller| 2016-03-04|f77e500e7a|7cbcc791bf|
| 58776|    2017-01-01|              49.0|            Vendetta|       Action, Drama| 2015-06-12|c74aec7673|ebf43c36b6|
| 58777|    2017-01-01|               0.0|The SpongeBob Squ...|Animation, Action...| 2004-11-19|a80d6fc2aa|a57c992287|
+------+--------------+------------------+------

In [8]:
df = df.withColumn(
    "calculated_time_to_next",
    fun.lag("click_datetime")
)
df.show(5)

AnalysisException: [WINDOW_FUNCTION_WITHOUT_OVER_CLAUSE] Window function "lag(click_datetime, 1, NULL)" requires an OVER clause.;
Project [row_id#116, click_datetime#117, time_to_next_click#118, movie_title#119, movie_genres#120, relase_date#121, totle_id#122, user_id#123, lag(click_datetime#117, -1, null) AS calculated_time_to_next#175]
+- Relation [row_id#116,click_datetime#117,time_to_next_click#118,movie_title#119,movie_genres#120,relase_date#121,totle_id#122,user_id#123] csv
